# Singleton warbles vs warbles-in-bouts

Compare warble calls that occur alone versus warble calls that are part of a multi-call bout.

**Bout definition (same as `high_freq_bouts_v3.ipynb`):** within `(date_folder, exp, assigned_location)`, two consecutive warbles belong to the same bout if their silent gap (`this.start - prev.stop`) is in `[MIN_ICI_BOUT_S, MAX_ICI_BOUT_S]`. Gaps outside that window start a new bout.

**Singleton** = warble whose bout has size 1.
**In-bout** = warble whose bout has size ≥ `MIN_BOUT_SIZE` (default 5).

Warbles in 2–4-call bouts are dropped from this comparison so the two groups are well-separated.

Comparisons so far:
- **call duration** (works on Mac and cluster — uses table columns only)
- **peak / end frequency** via librosa STFT (cluster only — needs raw WAVs)

## Setup (cross-platform)

In [ ]:
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Single source of truth for bout-detection thresholds (see
# vocalization_analysis/bouts.py). If you want to tune them, do it there.
from vocalization_analysis.bouts import BOUT_THRESHOLDS, detect_bouts

HOST = platform.system()

if HOST == "Darwin":
    DROPBOX     = Path("/Users/gilyginosar/Dropbox (Personal)/Vocalizations_project")
    PARQUET_DIR = DROPBOX / "Data" / "parquet_cache"
    FIGURES_DIR = DROPBOX / "Figures" / "warble_singletons_vs_bouts"
    SAVE_FIGS   = True
elif HOST == "Linux":
    PARQUET_DIR = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/all_calls/parquet_cache")
    FIGURES_DIR = None
    SAVE_FIGS   = False
else:
    raise RuntimeError(f"Unsupported platform: {HOST}")

if SAVE_FIGS:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name, fmt="pdf"):
    if not SAVE_FIGS:
        return
    fig.savefig(FIGURES_DIR / f"{name}.{fmt}", bbox_inches="tight")

DATES_TO_PLOT = ["2025_03", "2025_07", "2025_10", "2026_02"]

print(f"HOST                  = {HOST}")
print(f"PARQUET_DIR           = {PARQUET_DIR}")
print(f"SAVE_FIGS             = {SAVE_FIGS}")
print(f"warble thresholds     = {BOUT_THRESHOLDS['warble']}")

## Load warbles and assign bout ids

Pool all dates, restrict to `event_type == "warble"`, then run bout detection on each call's silent gap to its predecessor within `(date, exp, location)`. Every warble row ends up with `bout_id` and `bout_size`.

In [ ]:
# Step 1: load + filter to warbles.
parts = []
for date_tag in DATES_TO_PLOT:
    df_d = pd.read_parquet(PARQUET_DIR / f"all_calls_{date_tag}.parquet")
    df_d = df_d[df_d["event_type"] == "warble"]
    parts.append(df_d)
warble = pd.concat(parts, ignore_index=True)

# Step 2: bout detection (thresholds come from BOUT_THRESHOLDS["warble"]).
warble = detect_bouts(warble, "warble")

MIN_BOUT_SIZE = BOUT_THRESHOLDS["warble"]["min_bout_size"]  # for downstream prints / labels

print(f"{len(warble):,} warble calls total")
print()
print("Per-date breakdown (calls):")
print(warble.groupby(["date_folder", "bout_kind"]).size().unstack(fill_value=0))
print()
print(f"Per-date bout count (in_bout = size >= {MIN_BOUT_SIZE}):")
bout_sizes = warble.drop_duplicates("bout_id")[["date_folder", "bout_size"]]
print(
    bout_sizes.groupby("date_folder").agg(
        n_bouts=("bout_size", "size"),
        n_singleton_bouts=("bout_size", lambda s: (s == 1).sum()),
        n_small_bouts=("bout_size", lambda s: ((s >= 2) & (s < MIN_BOUT_SIZE)).sum()),
        n_in_bouts=("bout_size", lambda s: (s >= MIN_BOUT_SIZE).sum()),
    )
)

## Duration: singleton vs in-bout, per date

Per-date overlay of duration densities. "In-bout" pools all warble calls from bouts of size ≥ `MIN_BOUT_SIZE`, regardless of their position. We can split by position (1st vs later) in a follow-up cell if interesting.

In [ ]:
KIND_COLORS = {"singleton": "#457B9D", "in_bout": "#E76F51"}

# Drop a handful of obvious mis-segmentation outliers ( >= 1 s warble durations,
# same treatment as high_freq_bouts_v3.ipynb).
MAX_DUR_S = 1.0

dur = warble[["date_folder", "bout_kind", "duration_sec"]].copy()
n_total   = len(dur)
n_dropped = int((dur["duration_sec"] >= MAX_DUR_S).sum())
dur = dur[dur["duration_sec"] < MAX_DUR_S]
print(f"clipped {n_dropped:,} / {n_total:,} warble durations >= {MAX_DUR_S}s "
      f"({100*n_dropped/n_total:.3f}%)")

# Shared log-bins across dates so panels are visually comparable.
log_dur = np.log10(dur["duration_sec"].clip(lower=1e-3))
bins = np.linspace(log_dur.min(), log_dur.max(), 60)

fig, axes = plt.subplots(
    len(DATES_TO_PLOT), 1,
    figsize=(11, 2.4 * len(DATES_TO_PLOT)),
    sharex=True, sharey=False,
)
if len(DATES_TO_PLOT) == 1:
    axes = [axes]

for ax, date_tag in zip(axes, DATES_TO_PLOT):
    sub = dur[dur["date_folder"] == date_tag]
    if sub.empty:
        ax.set_axis_off()
        continue

    for kind_value in ("singleton", "in_bout"):
        s = sub[sub["bout_kind"] == kind_value]["duration_sec"]
        if s.empty:
            continue
        med = s.median() * 1000
        ax.hist(np.log10(s), bins=bins, density=True,
                color=KIND_COLORS[kind_value], edgecolor="white", linewidth=0.4,
                alpha=0.5,
                label=f"{kind_value} (n={len(s):,}, median={med:.0f} ms)")

    ax.set_title(f"{date_tag}   (n = {len(sub):,} calls)", loc="left", fontsize=10)
    ax.set_ylabel("Density")
    ax.legend(fontsize=9, loc="upper right")
    ax.tick_params(labelbottom=True)

# Time-unit tick labels.
tick_seconds = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
tick_labels  = ["10 ms", "20 ms", "50 ms", "100 ms", "200 ms", "500 ms", "1 s"]
for ax in axes:
    ax.set_xticks(np.log10(tick_seconds))
    ax.set_xticklabels(tick_labels, fontsize=9)

axes[-1].set_xlabel("Call duration")
fig.suptitle("Warble duration: singleton vs in-bout (per date)", y=1.0, fontsize=12)
fig.tight_layout()
save_fig(fig, "warble_duration_singleton_vs_inbout_per_date")
plt.show()

### Summary table

Per-date medians and 25/75 percentiles for the two groups, plus a Mann–Whitney U test (Bonferroni-corrected across the 4 dates).

In [ ]:
from scipy.stats import mannwhitneyu

N_TESTS = len(DATES_TO_PLOT)
rows = []
for date_tag in DATES_TO_PLOT:
    sub = dur[dur["date_folder"] == date_tag]
    s_single = sub.loc[sub["bout_kind"] == "singleton", "duration_sec"].values
    s_inbout = sub.loc[sub["bout_kind"] == "in_bout",   "duration_sec"].values
    if len(s_single) >= 3 and len(s_inbout) >= 3:
        _, p_raw = mannwhitneyu(s_single, s_inbout, alternative="two-sided")
        p_bonf = min(p_raw * N_TESTS, 1.0)
    else:
        p_raw = p_bonf = float("nan")
    rows.append({
        "date_folder":   date_tag,
        "n_singleton":   len(s_single),
        "n_in_bout":     len(s_inbout),
        "med_singleton_ms": round(1000 * np.median(s_single), 1) if len(s_single) else np.nan,
        "med_in_bout_ms":   round(1000 * np.median(s_inbout), 1) if len(s_inbout) else np.nan,
        "p25_singleton_ms": round(1000 * np.percentile(s_single, 25), 1) if len(s_single) else np.nan,
        "p75_singleton_ms": round(1000 * np.percentile(s_single, 75), 1) if len(s_single) else np.nan,
        "p25_in_bout_ms":   round(1000 * np.percentile(s_inbout, 25), 1) if len(s_inbout) else np.nan,
        "p75_in_bout_ms":   round(1000 * np.percentile(s_inbout, 75), 1) if len(s_inbout) else np.nan,
        "p_raw":   p_raw,
        "p_bonf":  p_bonf,
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## Acoustic features — peak vs end frequency (cluster only)

For each call we compute two per-call scalars from an STFT magnitude spectrogram:

- `peak_freq_hz` — frequency at the global energy maximum (sum across frames, then argmax). Single number for the whole call.
- `end_freq_hz`  — median of the per-frame peak-frequency over the **last `END_FRAC`** of frames. Captures how the call ends without being thrown by a single noisy frame.

Warbles are FM calls so the **peak vs end** difference is a rough measure of frequency modulation depth.

Runs on the cluster only (needs raw WAVs at `BASE_PROCESSED_AUDIO`). We subsample `N_PER_GROUP` calls per `(date, kind)` to keep runtime reasonable, and require `meanprob_warble >= WARBLE_PROB_THR` so we don't waste compute on iffy classifier hits.

In [ ]:
import functools
import soundfile as sf
import librosa

if HOST == "Linux":
    BASE_PROCESSED_AUDIO = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio")
else:
    BASE_PROCESSED_AUDIO = None  # raw WAVs aren't on the Mac

# Warble band — wider than the newborn band in explore_calls_xplatform so we
# don't clip the high end of warble peaks.
FREQ_BAND_HZ = (1_000, 60_000)
N_FFT        = 512    # ~4 ms at sr=125 kHz
HOP_LENGTH   = 128    # ~1 ms hop
END_FRAC     = 0.25   # fraction of frames at the end of the call to average for end_freq

def call_wav_path(date_folder, exp, channel, file_num):
    if BASE_PROCESSED_AUDIO is None:
        raise RuntimeError("Raw WAVs aren't accessible on this platform.")
    return (BASE_PROCESSED_AUDIO / date_folder / str(int(exp))
            / "Averaged_wavs_w_annotations"
            / f"channel_{int(channel)}_file_{int(file_num):03d}.wav")

@functools.lru_cache(maxsize=256)
def _wav_samplerate(path_str):
    return sf.info(path_str).samplerate

def load_call_slice(date_folder, exp, channel, file_num, start_sec, stop_sec):
    p = call_wav_path(date_folder, exp, channel, file_num)
    sr = _wav_samplerate(str(p))
    start = int(round(start_sec * sr))
    stop  = int(round(stop_sec  * sr))
    y, _ = sf.read(str(p), start=start, stop=stop, dtype="float32", always_2d=False)
    return y, sr

def peak_and_end_freq(y, sr, band=FREQ_BAND_HZ, n_fft=N_FFT, hop=HOP_LENGTH, end_frac=END_FRAC):
    """Return (peak_freq, end_freq) in Hz.

    peak_freq : frequency at the global STFT energy maximum (sum across frames).
    end_freq  : median per-frame peak frequency over the last `end_frac` of frames.
    Returns (nan, nan) if the call is too short or has no in-band energy.
    """
    if len(y) < n_fft:
        return np.nan, np.nan
    y = y - y.mean()
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop, window="hann"))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    in_band = (freqs >= band[0]) & (freqs <= band[1])
    if not in_band.any():
        return np.nan, np.nan
    S = S[in_band]
    f = freqs[in_band]
    if S.sum() == 0:
        return np.nan, np.nan

    # Global peak: collapse time, then argmax.
    peak = float(f[np.argmax(S.sum(axis=1))])

    # End-of-call peak: per-frame argmax over the last end_frac of frames, then median.
    n_frames = S.shape[1]
    n_end = max(1, int(round(end_frac * n_frames)))
    end_block = S[:, -n_end:]
    frame_energy = end_block.sum(axis=0)
    valid = frame_energy > 0
    if not valid.any():
        return peak, np.nan
    per_frame_peak = f[np.argmax(end_block[:, valid], axis=0)]
    end_freq = float(np.median(per_frame_peak))
    return peak, end_freq

In [ ]:
N_PER_GROUP     = 500     # warbles per (date_folder, bout_kind)
WARBLE_PROB_THR = 0.7     # require meanprob_warble >= this

if HOST != "Linux":
    raise RuntimeError("Acoustic features require raw WAVs — run this cell on the cluster.")

pool = warble[
    (warble["bout_kind"].isin(["singleton", "in_bout"]))
    & (warble["meanprob_warble"] >= WARBLE_PROB_THR)
].copy()
print(f"pool after meanprob_warble >= {WARBLE_PROB_THR}: {len(pool):,} calls")

# Balanced subsample: up to N_PER_GROUP per (date, bout_kind).
sub = (
    pool.groupby(["date_folder", "bout_kind"], group_keys=False)
        .apply(lambda g: g.sample(min(len(g), N_PER_GROUP), random_state=0))
        .reset_index(drop=True)
)
print(f"subsample: {len(sub):,} calls")
print(sub.groupby(["date_folder", "bout_kind"]).size().unstack(fill_value=0))

peaks = np.empty(len(sub))
ends  = np.empty(len(sub))
for i, row in enumerate(sub.itertuples(index=False)):
    y, sr = load_call_slice(row.date_folder, row.exp, row.channel,
                            row.file_num,
                            row.start_time_file_sec, row.stop_time_file_sec)
    peaks[i], ends[i] = peak_and_end_freq(y, sr)
    if (i + 1) % 500 == 0:
        print(f"  {i+1:,}/{len(sub):,}")

sub["peak_freq_hz"] = peaks
sub["end_freq_hz"]  = ends
sub["peak_minus_end_hz"] = sub["peak_freq_hz"] - sub["end_freq_hz"]

print()
print("Median per (date, bout_kind):")
print(
    sub.groupby(["date_folder", "bout_kind"]).agg(
        n=("peak_freq_hz", "count"),
        peak_khz=("peak_freq_hz", lambda s: round(s.median() / 1000, 2)),
        end_khz =("end_freq_hz",  lambda s: round(s.median() / 1000, 2)),
        peak_minus_end_khz=("peak_minus_end_hz", lambda s: round(s.median() / 1000, 2)),
    )
)

In [ ]:
# Per-date overlay: peak_freq (top row) and end_freq (bottom row),
# singleton vs in_bout. Shared bins per row so panels are comparable.
FEATURES = [
    ("peak_freq_hz", "Peak frequency"),
    ("end_freq_hz",  "End frequency"),
]

fig, axes = plt.subplots(
    len(FEATURES), len(DATES_TO_PLOT),
    figsize=(3.2 * len(DATES_TO_PLOT), 3.2 * len(FEATURES)),
    sharex="row", sharey="row",
)

for r, (col, label) in enumerate(FEATURES):
    vals_all = sub[col].dropna() / 1000   # kHz
    if vals_all.empty:
        continue
    bins = np.linspace(vals_all.min(), vals_all.max(), 45)

    for c, date_tag in enumerate(DATES_TO_PLOT):
        ax = axes[r, c]
        d = sub[sub["date_folder"] == date_tag]
        for kind_value in ("singleton", "in_bout"):
            s = d[d["bout_kind"] == kind_value][col].dropna() / 1000
            if s.empty:
                continue
            ax.hist(s, bins=bins, density=True,
                    color=KIND_COLORS[kind_value], edgecolor="white", linewidth=0.4,
                    alpha=0.5,
                    label=f"{kind_value} (n={len(s):,}, med={s.median():.1f} kHz)")
        if r == 0:
            ax.set_title(date_tag, fontsize=10)
        if c == 0:
            ax.set_ylabel(f"{label}\nDensity", fontsize=10)
        if r == len(FEATURES) - 1:
            ax.set_xlabel("kHz")
        ax.legend(fontsize=7, loc="upper right")

fig.suptitle(
    f"Warble peak vs end frequency  (n ≤ {N_PER_GROUP}/group, "
    f"meanprob_warble ≥ {WARBLE_PROB_THR})",
    y=1.0, fontsize=12,
)
fig.tight_layout()
save_fig(fig, f"warble_peak_end_freq_singleton_vs_inbout_thr{int(WARBLE_PROB_THR*100)}")
plt.show()

## Warble context: stack-followed vs deep-in-warble-bout

Split warbles into two acoustically-interesting subpopulations:

- **`inword`**: warble whose **immediately following call is a `stacks`** within 50 ms. (i.e. the warble is the first half of a warble→stacks "word" pattern.)
- **`bout_member`**: warble classified as `in_bout` by warble-bout detection, AND no non-warble neighbor within 50 ms in either direction. So the warble is "deep" in a warble-only sequence.

After this cell, every row of `warble` has a `context_class` column and the diagnostic fields `next_call_type`, `next_call_gap_s`, `nonwarble_before_s`, `nonwarble_after_s` for inspection. Use `warble[warble["context_class"]=="inword"]` etc. to find specific calls for video lookup.

In [ ]:
# Classify warbles by surrounding context.
# - inword: the very next call after this warble is a "stacks" within 50 ms.
# - bout_member: classified as in_bout by detect_bouts AND no non-warble within
#   50 ms on either side.

INWORD_GAP_S = 0.050   # max gap to the following stacks call
GROUP_COLS   = ["date_folder", "exp", "assigned_location"]
NS_PER_S = 1_000_000_000

# Load full corpus (warbles + non-warbles).
parts_all = []
for date_tag in DATES_TO_PLOT:
    df_d = pd.read_parquet(PARQUET_DIR / f"all_calls_{date_tag}.parquet")
    parts_all.append(df_d)
all_calls = pd.concat(parts_all, ignore_index=True)
print(f"All calls loaded: {len(all_calls):,}")
print(all_calls["event_type"].value_counts().to_string())

# Convert times to int64 ns for fast NumPy arithmetic.
warble = warble.copy()
warble["_start_ns"] = warble["start_time_real"].astype("int64")
warble["_stop_ns"]  = warble["stop_time_real"].astype("int64")
all_calls["_start_ns"] = all_calls["start_time_real"].astype("int64")
all_calls["_stop_ns"]  = all_calls["stop_time_real"].astype("int64")

# Per-group sorted arrays:
#   - non-warble start/stop  (for the bout_member diagnostic)
#   - ALL calls' starts + their types  (for the inword next-call lookup)
nonwarble_data    = {}
all_calls_by_grp  = {}
for keys, gdf in all_calls.groupby(GROUP_COLS):
    sort_by_start = np.argsort(gdf["_start_ns"].values)
    all_calls_by_grp[keys] = {
        "starts": gdf["_start_ns"].values[sort_by_start],
        "types":  gdf["event_type"].values[sort_by_start],
    }
    nw_mask = gdf["event_type"] != "warble"
    if nw_mask.any():
        nonwarble_data[keys] = {
            "stops":  np.sort(gdf.loc[nw_mask, "_stop_ns"].values),
            "starts": np.sort(gdf.loc[nw_mask, "_start_ns"].values),
        }

# Initialise per-warble columns.
warble["nonwarble_before_s"] = np.nan
warble["nonwarble_after_s"]  = np.nan
warble["next_call_type"]     = ""
warble["next_call_gap_s"]    = np.nan

for keys, w_gdf in warble.groupby(GROUP_COLS):
    idx      = w_gdf.index.values
    w_starts = w_gdf["_start_ns"].values
    w_stops  = w_gdf["_stop_ns"].values

    # --- nearest non-warble neighbour (bout_member diagnostic) ---
    if keys in nonwarble_data:
        nw_stops  = nonwarble_data[keys]["stops"]
        nw_starts = nonwarble_data[keys]["starts"]
        i = np.searchsorted(nw_stops, w_starts, side="right") - 1
        v_b = i >= 0
        warble.loc[idx, "nonwarble_before_s"] = np.where(
            v_b, (w_starts - nw_stops[np.clip(i, 0, None)]) / NS_PER_S, np.nan,
        )
        j = np.searchsorted(nw_starts, w_stops, side="left")
        v_a = j < len(nw_starts)
        warble.loc[idx, "nonwarble_after_s"] = np.where(
            v_a, (nw_starts[np.clip(j, 0, len(nw_starts) - 1)] - w_stops) / NS_PER_S, np.nan,
        )

    # --- next call (any type) immediately after this warble ---
    if keys in all_calls_by_grp:
        starts = all_calls_by_grp[keys]["starts"]
        types  = all_calls_by_grp[keys]["types"]
        # Find first call whose start > warble's stop. Use side="right" to skip
        # any call that ends at exactly w_stop (i.e. the warble itself).
        k = np.searchsorted(starts, w_stops, side="right")
        has_next = k < len(starts)
        warble.loc[idx, "next_call_type"] = np.where(
            has_next, types[np.clip(k, 0, len(types) - 1)], "",
        )
        warble.loc[idx, "next_call_gap_s"] = np.where(
            has_next,
            (starts[np.clip(k, 0, len(starts) - 1)] - w_stops) / NS_PER_S,
            np.nan,
        )

warble["nearest_nonwarble_s"] = warble[["nonwarble_before_s",
                                        "nonwarble_after_s"]].min(axis=1)

# --- Classification ---
inword_mask = (
    (warble["next_call_type"] == "stacks")
    & (warble["next_call_gap_s"] < INWORD_GAP_S)
)
bout_mask = (
    (warble["bout_kind"] == "in_bout")
    & (warble["nearest_nonwarble_s"] >= INWORD_GAP_S)
)
warble["context_class"] = "other"
warble.loc[inword_mask, "context_class"] = "inword"
# bout_mask must NOT overlap with inword (in case a warble has stacks after AND
# is in_bout with no other non-warble neighbour - it's rare but possible).
warble.loc[bout_mask & ~inword_mask, "context_class"] = "bout_member"

print(f"\nInword: next call is `stacks` within {INWORD_GAP_S*1000:.0f} ms")
print("Counts per (date, class):")
print(warble.groupby(["date_folder", "context_class"]).size().unstack(fill_value=0))


### Extract acoustic features for the classified warbles

Subsample up to `N_PER_GROUP` warbles per (date, context_class), then run vocalpy's `compute_features` on each. Output goes into `warble_features_classified`.


In [ ]:
# Extract vocalpy features for inword + bout_member warbles.
from vocalization_analysis.acoustic_features import (
    compute_features as compute_vp_features,
    load_call_slice  as vp_load_call_slice,
)

if HOST != "Linux":
    raise RuntimeError("Feature extraction needs raw WAVs — run on the cluster.")

N_PER_GROUP = 300
PAD_SEC     = 0.01

classified = warble[warble["context_class"].isin(["inword", "bout_member"])].copy()
sub_class = (
    classified.groupby(["date_folder", "context_class"], group_keys=False)
              .apply(lambda g: g.sample(min(len(g), N_PER_GROUP), random_state=0))
              .reset_index(drop=True)
)
print(f"Sampled {len(sub_class):,} warbles")
print(sub_class.groupby(["date_folder", "context_class"]).size().unstack(fill_value=0))

records = []
for i, row in enumerate(sub_class.itertuples(index=False)):
    try:
        y, sr = vp_load_call_slice(
            BASE_PROCESSED_AUDIO,
            row.date_folder, row.exp, row.channel, row.file_num,
            row.start_time_file_sec, row.stop_time_file_sec,
            pad_sec=PAD_SEC,
        )
        feats = compute_vp_features(y, sr)
        feats["duration_s"] = row.stop_time_file_sec - row.start_time_file_sec
        records.append({
            "date_folder":   row.date_folder,
            "context_class": row.context_class,
            "exp":           row.exp,
            "file_num":      row.file_num,
            "start_time_file_sec": row.start_time_file_sec,
            **feats,
        })
    except Exception as e:
        if len(records) < 5:
            print(f"  failed at i={i}: {e}")
    if (i + 1) % 500 == 0:
        print(f"  {i+1:,}/{len(sub_class):,}")

warble_features_classified = pd.DataFrame(records)
print(f"\nFeatures extracted: {len(warble_features_classified):,} rows")


### Compare features between inword and bout-member warbles

Per-feature × per-date violin. Mann-Whitney p (unpaired) and Cohen's d in each panel. Hypothesis from your qualitative read:

- **Inword shorter** (`duration_s`)
- **Inword less "warbly"** (lower `fm_median` — FM is the right operationalization)
- **Inword ends lower** (lower `stop_pitch_hz`)

Peak frequency included as a control — your qualitative description didn't predict a peak-frequency difference, so it should be flatter than the other three.


In [ ]:
from scipy.stats import mannwhitneyu

FEATURES_CMP = [
    ("Duration (ms)",                   "duration_s",    lambda v: v * 1000),
    ("FM median (rad) — 'warbliness'",  "fm_median",     lambda v: v),
    ("Stop pitch (kHz)",                "stop_pitch_hz", lambda v: v / 1000),
    ("Peak frequency (kHz)",            "peak_freq_hz",  lambda v: v / 1000),
]
CLASS_COLORS = {"inword": "#E76F51", "bout_member": "#2A9D8F"}

def stars(p):
    if np.isnan(p):  return ""
    if p < 0.001:    return "***"
    if p < 0.01:     return "**"
    if p < 0.05:     return "*"
    return "n.s."

fig, axes = plt.subplots(
    len(FEATURES_CMP), len(DATES_TO_PLOT),
    figsize=(2.7 * len(DATES_TO_PLOT), 2.6 * len(FEATURES_CMP)),
    sharey="row",
)

for r, (ylabel, col, scale_fn) in enumerate(FEATURES_CMP):
    for c, date_tag in enumerate(DATES_TO_PLOT):
        ax = axes[r, c]
        d = warble_features_classified[
            warble_features_classified["date_folder"] == date_tag
        ]
        inword_v = scale_fn(d.loc[d["context_class"] == "inword",      col].dropna().values)
        bout_v   = scale_fn(d.loc[d["context_class"] == "bout_member", col].dropna().values)
        if len(inword_v) < 5 or len(bout_v) < 5:
            ax.set_axis_off()
            continue

        parts = ax.violinplot([inword_v, bout_v], positions=[0, 1],
                              widths=0.75, showmeans=False, showmedians=True)
        for pc, color in zip(parts["bodies"],
                              [CLASS_COLORS["inword"], CLASS_COLORS["bout_member"]]):
            pc.set_facecolor(color); pc.set_alpha(0.5)

        try:
            _, p = mannwhitneyu(inword_v, bout_v, alternative="two-sided")
        except ValueError:
            p = np.nan
        n1, n2 = len(inword_v), len(bout_v)
        s1 = np.std(inword_v, ddof=1); s2 = np.std(bout_v, ddof=1)
        pooled = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
        d_val = (np.mean(bout_v) - np.mean(inword_v)) / pooled if pooled > 0 else np.nan

        ax.set_xticks([0, 1])
        ax.set_xticklabels(["inword", "bout"], fontsize=9)
        ax.set_xlim(-0.6, 1.6)
        if c == 0:
            ax.set_ylabel(ylabel, fontsize=9)
        if r == 0:
            ax.set_title(f"{date_tag}  (n={n1}/{n2})", fontsize=9)
        ax.text(0.5, 0.97,
                f"d = {d_val:+.2f}\np = {p:.1e} {stars(p)}",
                transform=ax.transAxes, ha="center", va="top", fontsize=8)

fig.suptitle(
    "Warble features: inword (non-warble within 50 ms) vs deep-in-bout",
    y=1.02, fontsize=12,
)
fig.tight_layout()
save_fig(fig, "warble_inword_vs_bout_features")
plt.show()


### Sanity check: spectrogram examples per class

Visualize a sample from each class with their surrounding audio context. Each panel:
- 1-second window centered on the target warble
- Spectrogram
- Colored vertical lines mark **every** call boundary in that window (solid = start, dashed = stop). Color encodes call type — white = the target warble itself, other colors = other calls (alarm/high-freq/stacks/etc).

For **`inword`** examples you should see a non-warble (color ≠ white) very close to the target warble. For **`bout_member`** examples you should see only warbles (white) nearby, and the nearest non-warble — if visible at all — should be off to the edges of the window.

Use this to verify the detection logic and to pick examples for video review.

In [ ]:
# Sanity-check examples for inword vs bout_member warbles.
import librosa
import matplotlib.patches as mpatches

if HOST != "Linux":
    raise RuntimeError("Spectrograms need raw WAVs - run on the cluster.")

N_PER_CLASS = 8
WINDOW_SEC  = 1.0     # total window around each example
NFFT_PLOT   = 512
HOP_PLOT    = 128

CALL_TYPE_COLORS = {
    "warble":     "white",
    "alarm":      "red",
    "high-freq":  "orange",
    "stacks":     "lime",
    "noise":      "gray",
}

# Sample examples.
inword_pool   = warble[warble["context_class"] == "inword"]
bout_pool     = warble[warble["context_class"] == "bout_member"]
inword_ex = inword_pool.sample(min(N_PER_CLASS, len(inword_pool)), random_state=1)
bout_ex   = bout_pool.sample(min(N_PER_CLASS,   len(bout_pool)),   random_state=1)

fig, axes = plt.subplots(
    N_PER_CLASS, 2,
    figsize=(13, 2.0 * N_PER_CLASS),
)

for col_i, (class_name, examples) in enumerate(
    [("inword", inword_ex), ("bout_member", bout_ex)]
):
    for row_i, (_, row) in enumerate(examples.iterrows()):
        ax = axes[row_i, col_i]
        call_mid  = (row["start_time_file_sec"] + row["stop_time_file_sec"]) / 2
        win_start = max(0.0, call_mid - WINDOW_SEC / 2)
        win_stop  = call_mid + WINDOW_SEC / 2

        # Load audio for the window.
        try:
            y, sr = vp_load_call_slice(
                BASE_PROCESSED_AUDIO,
                row["date_folder"], row["exp"], row["channel"], row["file_num"],
                win_start, win_stop, pad_sec=0.0,
            )
        except Exception as e:
            ax.text(0.5, 0.5, f"load failed:\n{e}", transform=ax.transAxes,
                    ha="center", va="center", fontsize=8)
            ax.set_axis_off()
            continue
        actual_dur = len(y) / sr
        win_stop_actual = win_start + actual_dur

        # Spectrogram.
        if len(y) < NFFT_PLOT:
            ax.text(0.5, 0.5, "too short", transform=ax.transAxes,
                    ha="center", va="center", fontsize=8)
            ax.set_axis_off()
            continue
        S    = np.abs(librosa.stft(y.astype(np.float32),
                                    n_fft=NFFT_PLOT, hop_length=HOP_PLOT, window="hann"))
        S_db = librosa.amplitude_to_db(S, ref=np.max)
        freqs = librosa.fft_frequencies(sr=sr, n_fft=NFFT_PLOT)
        ax.imshow(
            S_db, aspect="auto", origin="lower", cmap="magma", vmin=-60, vmax=0,
            extent=[win_start, win_stop_actual, freqs[0] / 1000, freqs[-1] / 1000],
        )
        ax.set_ylim(1, 60)

        # Overlay all call boundaries in the window.
        in_win = all_calls[
            (all_calls["date_folder"] == row["date_folder"])
            & (all_calls["exp"]       == row["exp"])
            & (all_calls["channel"]   == row["channel"])
            & (all_calls["file_num"]  == row["file_num"])
            & (all_calls["start_time_file_sec"] < win_stop_actual)
            & (all_calls["stop_time_file_sec"]  > win_start)
        ]
        for _, oc in in_win.iterrows():
            color = CALL_TYPE_COLORS.get(oc["event_type"], "cyan")
            is_target = (
                (abs(oc["start_time_file_sec"] - row["start_time_file_sec"]) < 1e-6)
                and (oc["event_type"] == "warble")
            )
            lw = 2.5 if is_target else 1.2
            alpha = 0.95 if is_target else 0.7
            ax.axvline(oc["start_time_file_sec"], color=color,
                       linewidth=lw, alpha=alpha)
            ax.axvline(oc["stop_time_file_sec"],  color=color,
                       linewidth=lw, alpha=alpha, linestyle="--")

        # Title with key info.
        nb_b = row.get("nonwarble_before_s")
        nb_a = row.get("nonwarble_after_s")
        def fmt_ms(x):
            return "—" if pd.isna(x) else f"{x*1000:.0f} ms"
        ax.set_title(
            f"{class_name}  |  {row['date_folder']}  exp{int(row['exp'])}/f{int(row['file_num']):03d}\n"
            f"non-warble before: {fmt_ms(nb_b)},  after: {fmt_ms(nb_a)}",
            fontsize=8,
        )
        if col_i == 0:
            ax.set_ylabel("kHz", fontsize=8)

axes[-1, 0].set_xlabel("Time in file (s)", fontsize=9)
axes[-1, 1].set_xlabel("Time in file (s)", fontsize=9)

# Legend for call-type colors.
legend_elements = [
    mpatches.Patch(color=c, label=("target warble" if t == "warble" else t))
    for t, c in CALL_TYPE_COLORS.items()
]
fig.legend(handles=legend_elements, loc="upper center", ncol=len(CALL_TYPE_COLORS),
           bbox_to_anchor=(0.5, 1.00), fontsize=9)

fig.suptitle("Inword (left) vs deep-in-bout (right) — verify detection",
             y=1.02, fontsize=12)
fig.tight_layout()
save_fig(fig, "warble_context_examples_spectrograms")
plt.show()


### Hand-pick inword candidates

Plot a numbered grid of `inword`-classified warbles for visual curation. The
sampled set is stored as `inword_candidates_for_review` (with a stable order
defined by `RANDOM_STATE`), so the number you see in the upper-left of each
panel corresponds to that row's position.

Workflow:

1. Run this cell.
2. Look through the panels and write down the numbers of the ones that *really*
   look like a warble→stacks word (e.g. "the good ones are 3, 7, 14, 22").
3. Use `inword_candidates_for_review.iloc[[n-1 for n in good_numbers]]` in a
   downstream cell to pull just the hand-picked good ones (numbers are
   1-indexed in the plot, so subtract 1 to index).


In [ ]:
# Hand-pick inword candidates: numbered grid of spectrograms with call boundaries.
import librosa
import matplotlib.patches as mpatches

if HOST != "Linux":
    raise RuntimeError("Spectrograms need raw WAVs — run on the cluster.")

N_CANDIDATES = 30           # how many panels to show (tune as needed)
N_COLS       = 5            # grid columns
WINDOW_SEC   = 1.0
NFFT_PLOT    = 512
HOP_PLOT     = 128
RANDOM_STATE = 7            # change to get a different batch

CALL_TYPE_COLORS = {
    "warble":     "white",
    "alarm":      "red",
    "high-freq":  "orange",
    "stacks":     "lime",
    "noise":      "gray",
}

inword_pool = warble[warble["context_class"] == "inword"]
if len(inword_pool) == 0:
    raise RuntimeError("No inword candidates - run the classification cell first.")

# Stable sample (saved to a variable the user can index later).
n_take = min(N_CANDIDATES, len(inword_pool))
inword_candidates_for_review = (
    inword_pool.sample(n_take, random_state=RANDOM_STATE).reset_index(drop=False)
)
print(f"Sampled {n_take} inword candidates "
      f"(random_state={RANDOM_STATE}). "
      f"Numbers in the plot are 1..{n_take}; "
      f"use inword_candidates_for_review.iloc[[n-1 for n in good_numbers]] to retrieve.")

n_rows = (n_take + N_COLS - 1) // N_COLS
fig, axes = plt.subplots(n_rows, N_COLS,
                         figsize=(3.2 * N_COLS, 2.2 * n_rows),
                         squeeze=False)

for i in range(n_rows * N_COLS):
    r, c = divmod(i, N_COLS)
    ax = axes[r, c]
    if i >= n_take:
        ax.set_axis_off()
        continue
    row = inword_candidates_for_review.iloc[i]

    call_mid  = (row["start_time_file_sec"] + row["stop_time_file_sec"]) / 2
    win_start = max(0.0, call_mid - WINDOW_SEC / 2)
    win_stop  = call_mid + WINDOW_SEC / 2

    try:
        y, sr = vp_load_call_slice(
            BASE_PROCESSED_AUDIO,
            row["date_folder"], row["exp"], row["channel"], row["file_num"],
            win_start, win_stop, pad_sec=0.0,
        )
    except Exception as e:
        ax.text(0.5, 0.5, f"load failed:\n{e}", transform=ax.transAxes,
                ha="center", va="center", fontsize=7)
        ax.set_axis_off()
        continue
    if len(y) < NFFT_PLOT:
        ax.text(0.5, 0.5, "too short", transform=ax.transAxes,
                ha="center", va="center", fontsize=7)
        ax.set_axis_off()
        continue

    actual_dur      = len(y) / sr
    win_stop_actual = win_start + actual_dur

    S    = np.abs(librosa.stft(y.astype(np.float32),
                                n_fft=NFFT_PLOT, hop_length=HOP_PLOT, window="hann"))
    S_db = librosa.amplitude_to_db(S, ref=np.max)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=NFFT_PLOT)
    ax.imshow(
        S_db, aspect="auto", origin="lower", cmap="magma", vmin=-60, vmax=0,
        extent=[win_start, win_stop_actual, freqs[0] / 1000, freqs[-1] / 1000],
    )
    ax.set_ylim(1, 60)

    # Overlay call boundaries.
    in_win = all_calls[
        (all_calls["date_folder"] == row["date_folder"])
        & (all_calls["exp"]       == row["exp"])
        & (all_calls["channel"]   == row["channel"])
        & (all_calls["file_num"]  == row["file_num"])
        & (all_calls["start_time_file_sec"] < win_stop_actual)
        & (all_calls["stop_time_file_sec"]  > win_start)
    ]
    for _, oc in in_win.iterrows():
        color = CALL_TYPE_COLORS.get(oc["event_type"], "cyan")
        is_target = (
            (abs(oc["start_time_file_sec"] - row["start_time_file_sec"]) < 1e-6)
            and (oc["event_type"] == "warble")
        )
        lw    = 2.5 if is_target else 1.2
        alpha = 0.95 if is_target else 0.7
        ax.axvline(oc["start_time_file_sec"], color=color,
                   linewidth=lw, alpha=alpha)
        ax.axvline(oc["stop_time_file_sec"],  color=color,
                   linewidth=lw, alpha=alpha, linestyle="--")

    # Big number badge (1-indexed for human reference).
    ax.text(
        0.03, 0.95, f"{i+1}",
        transform=ax.transAxes,
        fontsize=18, fontweight="bold",
        color="white", va="top", ha="left",
        bbox=dict(facecolor="black", alpha=0.6, pad=2, edgecolor="none"),
    )

    # Concise sub-title with key diagnostic.
    gap_ms = row["next_call_gap_s"] * 1000 if not pd.isna(row["next_call_gap_s"]) else None
    ax.set_title(
        f"{row['date_folder']} exp{int(row['exp'])}/f{int(row['file_num']):03d}  "
        f"→{row['next_call_type']} in {gap_ms:.0f}ms" if gap_ms is not None
        else f"{row['date_folder']} exp{int(row['exp'])}/f{int(row['file_num']):03d}",
        fontsize=8,
    )
    if c == 0:
        ax.set_ylabel("kHz", fontsize=8)

# Legend.
legend_elements = [
    mpatches.Patch(color=col, label=("target warble" if t == "warble" else t))
    for t, col in CALL_TYPE_COLORS.items()
]
fig.legend(handles=legend_elements, loc="upper center",
           ncol=len(CALL_TYPE_COLORS), bbox_to_anchor=(0.5, 1.005), fontsize=9)

fig.suptitle(f"Inword candidates for hand curation  ·  N={n_take}",
             y=1.02, fontsize=12)
fig.tight_layout()
plt.show()

# Print the table so they have it for reference.
print("\nCandidate table (1-indexed in plot, 0-indexed in DataFrame):")
print(inword_candidates_for_review[["date_folder", "exp", "file_num",
                                    "start_time_file_sec", "next_call_type",
                                    "next_call_gap_s"]].to_string())


### Hand-labelling workflow: batch viewer

Loop through inword candidates in reproducible batches. The same `BATCH_NUM` always shows the same 30 warbles, so you can resume across sessions.

Workflow:
1. Set `BATCH_NUM = 0`, run **this cell** (the viewer). Look at the grid.
2. In the **next cell** (the recorder), set `GOOD_NUMBERS_THIS_BATCH = [3, 7, 14, ...]` and run it — those warbles get appended to `warble_inword_labels.csv`.
3. Bump `BATCH_NUM` by 1, repeat. The CSV survives kernel restarts.

In [ ]:
# Batch viewer for hand-labelling inword warbles.
import librosa
import matplotlib.patches as mpatches

if HOST != "Linux":
    raise RuntimeError("Spectrograms need raw WAVs - run on the cluster.")

# === Knobs ===
BATCH_NUM   = 0            # change this each batch; same BATCH_NUM always shows the same 30
BATCH_SIZE  = 30
N_COLS      = 5
WINDOW_SEC  = 1.0
NFFT_PLOT   = 512
HOP_PLOT    = 128

# Build a STABLE ordering of all inword candidates so BATCH_NUM is reproducible.
inword_all = warble[warble["context_class"] == "inword"].copy()
inword_all["uid"] = (
    inword_all["date_folder"].astype(str) + "_"
    + inword_all["exp"].astype(int).astype(str) + "_"
    + inword_all["channel"].astype(int).astype(str) + "_"
    + inword_all["file_num"].astype(int).astype(str).str.zfill(3) + "_"
    + (inword_all["start_time_file_sec"] * 1000).round().astype(int).astype(str)
)
inword_all = inword_all.sort_values("uid").reset_index(drop=True)
print(f"Total inword candidates: {len(inword_all)}")
print(f"Total batches at BATCH_SIZE={BATCH_SIZE}: {(len(inword_all) + BATCH_SIZE - 1) // BATCH_SIZE}")

start = BATCH_NUM * BATCH_SIZE
end   = min(start + BATCH_SIZE, len(inword_all))
current_batch = inword_all.iloc[start:end].reset_index(drop=True)
current_batch["batch_num"]     = BATCH_NUM
current_batch["candidate_num"] = range(1, len(current_batch) + 1)
print(f"Batch {BATCH_NUM}: candidates {start}..{end-1}  ({len(current_batch)} shown)")

CALL_TYPE_COLORS = {
    "warble":     "white",
    "alarm":      "red",
    "high-freq":  "orange",
    "stacks":     "lime",
    "noise":      "gray",
}

n_rows = (len(current_batch) + N_COLS - 1) // N_COLS
fig, axes = plt.subplots(n_rows, N_COLS,
                         figsize=(3.2 * N_COLS, 2.2 * n_rows),
                         squeeze=False)

for i in range(n_rows * N_COLS):
    r, c = divmod(i, N_COLS)
    ax = axes[r, c]
    if i >= len(current_batch):
        ax.set_axis_off()
        continue
    row = current_batch.iloc[i]

    call_mid  = (row["start_time_file_sec"] + row["stop_time_file_sec"]) / 2
    win_start = max(0.0, call_mid - WINDOW_SEC / 2)
    win_stop  = call_mid + WINDOW_SEC / 2

    try:
        y, sr = vp_load_call_slice(
            BASE_PROCESSED_AUDIO,
            row["date_folder"], row["exp"], row["channel"], row["file_num"],
            win_start, win_stop, pad_sec=0.0,
        )
    except Exception as e:
        ax.text(0.5, 0.5, f"load failed:\n{e}", transform=ax.transAxes,
                ha="center", va="center", fontsize=7)
        ax.set_axis_off()
        continue
    if len(y) < NFFT_PLOT:
        ax.set_axis_off(); continue

    actual_dur      = len(y) / sr
    win_stop_actual = win_start + actual_dur

    S    = np.abs(librosa.stft(y.astype(np.float32),
                                n_fft=NFFT_PLOT, hop_length=HOP_PLOT, window="hann"))
    S_db = librosa.amplitude_to_db(S, ref=np.max)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=NFFT_PLOT)
    ax.imshow(S_db, aspect="auto", origin="lower", cmap="magma", vmin=-60, vmax=0,
              extent=[win_start, win_stop_actual, freqs[0] / 1000, freqs[-1] / 1000])
    ax.set_ylim(1, 60)

    in_win = all_calls[
        (all_calls["date_folder"] == row["date_folder"])
        & (all_calls["exp"]       == row["exp"])
        & (all_calls["channel"]   == row["channel"])
        & (all_calls["file_num"]  == row["file_num"])
        & (all_calls["start_time_file_sec"] < win_stop_actual)
        & (all_calls["stop_time_file_sec"]  > win_start)
    ]
    for _, oc in in_win.iterrows():
        color = CALL_TYPE_COLORS.get(oc["event_type"], "cyan")
        is_target = (
            (abs(oc["start_time_file_sec"] - row["start_time_file_sec"]) < 1e-6)
            and (oc["event_type"] == "warble")
        )
        lw    = 2.5 if is_target else 1.2
        alpha = 0.95 if is_target else 0.7
        ax.axvline(oc["start_time_file_sec"], color=color, linewidth=lw, alpha=alpha)
        ax.axvline(oc["stop_time_file_sec"],  color=color, linewidth=lw, alpha=alpha, linestyle="--")

    ax.text(0.03, 0.95, f"{i+1}",
            transform=ax.transAxes, fontsize=18, fontweight="bold",
            color="white", va="top", ha="left",
            bbox=dict(facecolor="black", alpha=0.6, pad=2, edgecolor="none"))

    gap_ms = row["next_call_gap_s"] * 1000 if not pd.isna(row["next_call_gap_s"]) else None
    title = (f"{row['date_folder']} exp{int(row['exp'])}/f{int(row['file_num']):03d}"
             + (f"  →{row['next_call_type']} in {gap_ms:.0f}ms" if gap_ms is not None else ""))
    ax.set_title(title, fontsize=8)

legend_elements = [
    mpatches.Patch(color=col, label=("target warble" if t == "warble" else t))
    for t, col in CALL_TYPE_COLORS.items()
]
fig.legend(handles=legend_elements, loc="upper center",
           ncol=len(CALL_TYPE_COLORS), bbox_to_anchor=(0.5, 1.005), fontsize=9)

fig.suptitle(f"Batch {BATCH_NUM}  ·  candidates {start+1}..{end}  "
             f"(of {len(inword_all)} total inword)",
             y=1.02, fontsize=12)
fig.tight_layout()
plt.show()


### Hand-labelling workflow: recorder

After looking at the batch above, fill in `GOOD_NUMBERS_THIS_BATCH` with the numbers of the good warbles and run this cell. The selected warbles get appended to `warble_inword_labels.csv` (idempotent — same UID never added twice).

After enough batches, load the final labeled set with:
```python
labels = pd.read_csv(LABEL_CSV_PATH)
```
and use it for downstream feature comparisons.

In [ ]:
# Recorder: append good warbles from the current batch to a persistent CSV.

# === User input: which numbers in the batch above are good? ===
GOOD_NUMBERS_THIS_BATCH = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]

# Persistent label storage (lives next to the notebook).
LABEL_CSV_PATH = Path("warble_inword_labels.csv")

LABEL_COLS = [
    "uid", "batch_num", "candidate_num",
    "date_folder", "exp", "channel", "file_num",
    "start_time_file_sec", "stop_time_file_sec",
    "next_call_type", "next_call_gap_s",
]

# Load existing labels.
if LABEL_CSV_PATH.exists():
    labels_df = pd.read_csv(LABEL_CSV_PATH)
else:
    labels_df = pd.DataFrame(columns=LABEL_COLS)

# Append this batch's selections.
if GOOD_NUMBERS_THIS_BATCH:
    sel = current_batch.iloc[[n - 1 for n in GOOD_NUMBERS_THIS_BATCH]].copy()
    new_rows = sel[LABEL_COLS]
    # Idempotent: skip UIDs already in the label file.
    new_rows = new_rows[~new_rows["uid"].isin(labels_df["uid"].values)]
    if not new_rows.empty:
        labels_df = pd.concat([labels_df, new_rows], ignore_index=True)
        labels_df.to_csv(LABEL_CSV_PATH, index=False)
        print(f"Added {len(new_rows)} new labels (skipped {len(sel) - len(new_rows)} dupes).")
    else:
        print("All selected UIDs were already in the label file - nothing new added.")
else:
    print("GOOD_NUMBERS_THIS_BATCH is empty - nothing to record.")

# Status.
print(f"\nTotal labeled so far: {len(labels_df)}")
print(f"Progress toward 300:  {100 * min(len(labels_df), 300) / 300:.0f}%")
print(f"Batches done (any labels):  "
      f"{labels_df['batch_num'].nunique() if len(labels_df) else 0}")
print(f"\nNext batch:  set BATCH_NUM = {BATCH_NUM + 1} above and re-run the viewer.")


## Hand-picked inword vs bout_member  ·  2025_03

Test whether the **302 hand-labeled** inword warbles (all from 2025_03) differ from bout_member warbles in the same date on the four features your qualitative read predicted:

- **Duration** — inword shorter
- **`fm_median`** — inword less "warbly"
- **`stop_pitch_hz`** — inword ends lower
- **`peak_freq_hz`** — control (no predicted difference)

If we see a clear, consistent contrast in 2025_03, we'll repeat for the other 3 dates.

In [ ]:
# Hand-picked inword (302 from 2025_03) vs bout_member (2025_03), feature comparison.

from scipy.stats import mannwhitneyu
from vocalization_analysis.acoustic_features import (
    compute_features as compute_vp_features,
    load_call_slice  as vp_load_call_slice,
)

if HOST != "Linux":
    raise RuntimeError("Feature extraction needs raw WAVs - run on the cluster.")

TARGET_DATE   = "2025_03"
LABEL_CSV     = Path("warble_inword_labels.csv")
N_BOUT_SAMPLE = 300   # subsample bout_member to a similar N
PAD_SEC       = 0.01

# --- Hand-picked inword set (from CSV, matched back to `warble` via UID) ---
labels_df = pd.read_csv(LABEL_CSV)
labels_2503 = labels_df[labels_df["date_folder"] == TARGET_DATE]

# Ensure `warble` has a `uid` column matching the recipe used by the
# batch viewer / recorder. Idempotent.
if "uid" not in warble.columns:
    warble["uid"] = (
        warble["date_folder"].astype(str) + "_"
        + warble["exp"].astype(int).astype(str) + "_"
        + warble["channel"].astype(int).astype(str) + "_"
        + warble["file_num"].astype(int).astype(str).str.zfill(3) + "_"
        + (warble["start_time_file_sec"] * 1000).round().astype(int).astype(str)
    )
good_inword = warble[
    warble["uid"].isin(labels_2503["uid"]) & (warble["date_folder"] == TARGET_DATE)
].copy()
print(f"Hand-picked inword warbles in {TARGET_DATE}: {len(good_inword)}")

# --- Date-matched bout_member sample ---
bout_pool = warble[
    (warble["context_class"] == "bout_member") & (warble["date_folder"] == TARGET_DATE)
]
n_bout = min(N_BOUT_SAMPLE, len(bout_pool))
bout_sample = bout_pool.sample(n_bout, random_state=0).copy()
print(f"Bout_member warbles sampled in {TARGET_DATE}: {n_bout} (pool: {len(bout_pool)})")

# --- Extract features for both groups ---
def extract_features(df, class_label):
    out = []
    for row in df.itertuples(index=False):
        try:
            y, sr = vp_load_call_slice(
                BASE_PROCESSED_AUDIO,
                row.date_folder, row.exp, row.channel, row.file_num,
                row.start_time_file_sec, row.stop_time_file_sec,
                pad_sec=PAD_SEC,
            )
            feats = compute_vp_features(y, sr)
            feats["duration_s"]    = row.stop_time_file_sec - row.start_time_file_sec
            feats["context_class"] = class_label
            feats["uid"]           = row.uid
            out.append(feats)
        except Exception:
            pass
    return out

print("\nExtracting features...")
recs_good = extract_features(good_inword,  "inword_handpicked")
recs_bout = extract_features(bout_sample,  "bout_member")
feat_df   = pd.DataFrame(recs_good + recs_bout)
print(f"Extracted: {len(feat_df)} rows  "
      f"({(feat_df['context_class']=='inword_handpicked').sum()} inword, "
      f"{(feat_df['context_class']=='bout_member').sum()} bout)")

# --- Plot: violins + Mann-Whitney + Cohen's d ---
FEATURES_CMP = [
    ("Duration (ms)",                 "duration_s",    lambda v: v * 1000),
    ("FM median (rad) — warbliness",  "fm_median",     lambda v: v),
    ("Stop pitch (kHz)",              "stop_pitch_hz", lambda v: v / 1000),
    ("Peak frequency (kHz)",          "peak_freq_hz",  lambda v: v / 1000),
]
CLASS_COLORS = {"inword_handpicked": "#E76F51", "bout_member": "#2A9D8F"}

def stars(p):
    if np.isnan(p):  return ""
    if p < 0.001:    return "***"
    if p < 0.01:     return "**"
    if p < 0.05:     return "*"
    return "n.s."

fig, axes = plt.subplots(1, len(FEATURES_CMP), figsize=(3.0 * len(FEATURES_CMP), 4))
for ax, (ylabel, col, scale_fn) in zip(axes, FEATURES_CMP):
    inword_v = scale_fn(feat_df.loc[feat_df["context_class"] == "inword_handpicked",
                                     col].dropna().values)
    bout_v   = scale_fn(feat_df.loc[feat_df["context_class"] == "bout_member",
                                     col].dropna().values)
    if len(inword_v) < 5 or len(bout_v) < 5:
        ax.set_axis_off(); continue

    parts = ax.violinplot([inword_v, bout_v], positions=[0, 1],
                          widths=0.75, showmeans=False, showmedians=True)
    for pc, color in zip(parts["bodies"],
                          [CLASS_COLORS["inword_handpicked"], CLASS_COLORS["bout_member"]]):
        pc.set_facecolor(color); pc.set_alpha(0.55)

    # Strip-scatter of individual points behind the violins.
    rng = np.random.default_rng(0)
    ax.scatter(rng.uniform(-0.12, 0.12, size=len(inword_v)), inword_v,
               s=6, alpha=0.35, color=CLASS_COLORS["inword_handpicked"], zorder=2)
    ax.scatter(1 + rng.uniform(-0.12, 0.12, size=len(bout_v)), bout_v,
               s=6, alpha=0.35, color=CLASS_COLORS["bout_member"], zorder=2)

    try:
        _, p = mannwhitneyu(inword_v, bout_v, alternative="two-sided")
    except ValueError:
        p = np.nan
    n1, n2 = len(inword_v), len(bout_v)
    s1, s2 = np.std(inword_v, ddof=1), np.std(bout_v, ddof=1)
    pooled = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
    d_val  = (np.mean(bout_v) - np.mean(inword_v)) / pooled if pooled > 0 else np.nan

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["inword\n(hand-picked)", "bout"], fontsize=9)
    ax.set_xlim(-0.6, 1.6)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(f"d = {d_val:+.2f}\np = {p:.1e} {stars(p)}\n(n={n1}/{n2})",
                 fontsize=9)

fig.suptitle(f"Hand-picked inword vs bout_member  ·  {TARGET_DATE}",
             y=1.02, fontsize=12)
fig.tight_layout()
save_fig(fig, f"warble_handpicked_inword_vs_bout_{TARGET_DATE}")
plt.show()

# Save the extracted features for any downstream use.
warble_handpicked_features = feat_df
print(f"\nfeature dataframe stored as `warble_handpicked_features` ({len(feat_df)} rows)")


## Video curation: export inword + bout_member examples to CSV

Build a flat CSV with everything needed to locate each candidate call in the video record:

- `date_folder`, `exp`, `assigned_location`
- `channel`, `file_num`
- `start_time_file_sec`, `stop_time_file_sec`, `duration_ms`
- `start_time_real` (absolute timestamp)
- `context_class` (`inword_handpicked` or `bout_member`)
- For inword: `next_call_type`, `next_call_gap_ms`, `next_call_start_time_file_sec`
- For bout: `bout_id`, `bout_position`, `bout_size`

Sorted by `date_folder, exp, file_num, start_time_file_sec` so you can scroll through one recording at a time.

In [ ]:
# Export inword + bout_member candidates to a CSV for video curation.

LABEL_CSV  = Path("warble_inword_labels.csv")
OUT_CSV    = Path("warble_video_curation.csv")
TARGET_DATE = "2025_03"      # match the date of the hand-labels
N_BOUT_OUT  = 302            # match the labelled inword N for a balanced review

# Ensure warble has a uid column (matches the labelling-pipeline recipe).
if "uid" not in warble.columns:
    warble["uid"] = (
        warble["date_folder"].astype(str) + "_"
        + warble["exp"].astype(int).astype(str) + "_"
        + warble["channel"].astype(int).astype(str) + "_"
        + warble["file_num"].astype(int).astype(str).str.zfill(3) + "_"
        + (warble["start_time_file_sec"] * 1000).round().astype(int).astype(str)
    )

# --- Hand-picked inword set ---
labels_df = pd.read_csv(LABEL_CSV)
labels_2503 = labels_df[labels_df["date_folder"] == TARGET_DATE]
inword_rows = warble[warble["uid"].isin(labels_2503["uid"])].copy()
inword_rows["context_class_export"] = "inword_handpicked"
print(f"inword_handpicked: {len(inword_rows)}")

# Find next-call start_time for the inword warbles too (some users want to see
# both the warble and the stacks that follows in the video).
all_calls["_start_ns"] = all_calls["start_time_real"].astype("int64")
GROUP_COLS = ["date_folder", "exp", "assigned_location"]
all_calls_sorted = {}
for keys, gdf in all_calls.groupby(GROUP_COLS):
    sort_idx = np.argsort(gdf["_start_ns"].values)
    all_calls_sorted[keys] = {
        "starts_sec_file": gdf["start_time_file_sec"].values[sort_idx],
        "starts_ns":       gdf["_start_ns"].values[sort_idx],
    }

inword_rows["next_call_start_time_file_sec"] = np.nan
inword_rows["_stop_ns"] = inword_rows["stop_time_real"].astype("int64")
for keys, w_gdf in inword_rows.groupby(GROUP_COLS):
    if keys not in all_calls_sorted:
        continue
    arr_ns   = all_calls_sorted[keys]["starts_ns"]
    arr_sec  = all_calls_sorted[keys]["starts_sec_file"]
    idx      = w_gdf.index.values
    w_stops  = w_gdf["_stop_ns"].values
    k = np.searchsorted(arr_ns, w_stops, side="right")
    has_next = k < len(arr_ns)
    inword_rows.loc[idx, "next_call_start_time_file_sec"] = np.where(
        has_next, arr_sec[np.clip(k, 0, len(arr_sec) - 1)], np.nan,
    )

# --- Matched bout_member sample (same date) ---
bout_pool = warble[
    (warble["context_class"] == "bout_member") & (warble["date_folder"] == TARGET_DATE)
]
n_bout = min(N_BOUT_OUT, len(bout_pool))
bout_rows = bout_pool.sample(n_bout, random_state=0).copy()
bout_rows["context_class_export"] = "bout_member"
bout_rows["next_call_start_time_file_sec"] = np.nan  # not relevant for bout export
print(f"bout_member (sampled): {len(bout_rows)} of {len(bout_pool)} available")

# --- Common columns ---
def coerce(df):
    df = df.copy()
    df["duration_ms"] = (df["stop_time_file_sec"] - df["start_time_file_sec"]) * 1000
    df["next_call_gap_ms"] = df["next_call_gap_s"] * 1000 if "next_call_gap_s" in df.columns else np.nan
    return df

inword_out = coerce(inword_rows)
bout_out   = coerce(bout_rows)

EXPORT_COLS = [
    "context_class_export",
    "date_folder", "exp", "assigned_location",
    "channel", "file_num",
    "start_time_file_sec", "stop_time_file_sec", "duration_ms",
    "start_time_real",
    # inword-specific
    "next_call_type", "next_call_gap_ms", "next_call_start_time_file_sec",
    # bout-specific
    "bout_id", "bout_position", "bout_size",
]

# Some columns might not exist on bout subset (e.g., next_call_type). Fill blanks.
for col in EXPORT_COLS:
    if col not in inword_out.columns:
        inword_out[col] = np.nan
    if col not in bout_out.columns:
        bout_out[col] = np.nan

combined = pd.concat([inword_out[EXPORT_COLS], bout_out[EXPORT_COLS]], ignore_index=True)
combined = combined.rename(columns={"context_class_export": "context_class"})

# Sort for efficient video review: one recording at a time.
combined = combined.sort_values(
    ["date_folder", "exp", "file_num", "start_time_file_sec"]
).reset_index(drop=True)

combined.to_csv(OUT_CSV, index=False)
print(f"\nWrote {len(combined)} rows -> {OUT_CSV.resolve()}")
print()
print("Per (context_class, file_num) breakdown - first 15 files:")
print(
    combined.groupby(["context_class", "file_num"])
            .size()
            .unstack(fill_value=0)
            .head(15)
)
print()
print("Sample rows:")
print(combined.head().to_string(index=False))


### Visual: location distribution between groups

Horizontal stacked bars showing the per-class breakdown of `assigned_location`. Each bar normalised to 100% within the class; the gap between the two bars is the headline finding — warble→stacks words happen almost entirely underground, while bout_member warbles are spread across arenas + underground.

In [ ]:
# Location distribution figure for the two warble groups.
# arena_1 + arena_2 merged into "above_ground".
from scipy.stats import chi2_contingency

df_loc = pd.read_csv(Path("warble_video_curation.csv"))

# Collapse arena_1 + arena_2 -> above_ground.
df_loc["location"] = df_loc["assigned_location"].replace(
    {"arena_1": "above_ground", "arena_2": "above_ground"}
)

ct        = df_loc.groupby(["context_class", "location"]).size().unstack(fill_value=0)
ct_pct    = (ct.T / ct.sum(axis=1)).T * 100
n_by_class = ct.sum(axis=1)

class_order = ["inword_handpicked", "bout_member"]
ct      = ct.reindex(class_order)
ct_pct  = ct_pct.reindex(class_order)

LOCATION_ORDER  = ["underground", "above_ground"]
LOCATION_COLORS = {
    "underground":  "#7B4F9A",
    "above_ground": "#E76F51",
}

chi2, p, dof, _ = chi2_contingency(ct[LOCATION_ORDER].values)

fig, ax = plt.subplots(figsize=(10, 3.2))
y_positions = np.arange(len(class_order))
left = np.zeros(len(class_order))

for loc in LOCATION_ORDER:
    if loc not in ct.columns:
        continue
    widths = ct_pct[loc].values
    counts = ct[loc].values
    ax.barh(
        y_positions, widths, left=left,
        label=loc, color=LOCATION_COLORS[loc],
        edgecolor="white", linewidth=1.0, height=0.6,
    )
    for yi, w, count, l_origin in zip(y_positions, widths, counts, left):
        if w >= 4:
            ax.text(
                l_origin + w / 2, yi,
                f"{int(count)}\n({w:.0f}%)",
                ha="center", va="center",
                color="white", fontsize=10, fontweight="bold",
            )
    left += widths

ax.set_yticks(y_positions)
ax.set_yticklabels([
    f"{cls}\n(n={n_by_class[cls]})" for cls in class_order
], fontsize=11)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("% within class", fontsize=11)
ax.set_title(
    f"Warble context by recording location  ·  2025_03\n"
    f"χ² = {chi2:.0f},   dof = {dof},   p = {p:.1e}   ***",
    fontsize=12,
)
ax.legend(
    loc="upper center", bbox_to_anchor=(0.5, -0.18),
    ncol=len(LOCATION_ORDER), frameon=False,
)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
save_fig(fig, "warble_context_by_location_collapsed_2025_03")
plt.show()


In [ ]:
# ---- Above-ground vs underground: alarm | warble (in_bout) | warble (inword) ----
# Assumes the inword cell above has run (sets warble["context_class"] and loads
# all_calls).

#CATEGORIES     = ["alarm", "warble\n(in bout)", "warble\n(in complex call)"]
CATEGORIES     = ["warble\n(in bout)", "warble\n(in complex call)"]

LOCATION_ORDER = ["underground", "arena"]
LOC_COLORS     = ["#E76F51", "#457B9D"]   # match LOCATION_ORDER

def _simplify_loc(s):
    return s.replace({"arena_1": "arena", "arena_2": "arena"})

#alarm    = all_calls.loc[all_calls["event_type"] == "alarm",    ["assigned_location"]].assign(category=CATEGORIES[0])
w_bout   = warble.loc[warble["context_class"] == "bout_member", ["assigned_location"]].assign(category=CATEGORIES[0])
w_inword = warble.loc[warble["context_class"] == "inword",      ["assigned_location"]].assign(category=CATEGORIES[1])

combined = pd.concat([w_bout, w_inword], ignore_index=True)
combined["location"] = _simplify_loc(combined["assigned_location"])

props = (
    combined.groupby(["category", "location"])
    .size()
    .unstack(fill_value=0)
    .reindex(CATEGORIES, fill_value=0)
    [LOCATION_ORDER]
)
totals     = props.sum(axis=1)
props_norm = props.div(totals, axis=0)

FS = 12

fig, ax = plt.subplots(figsize=(7, 5))
props_norm.plot.bar(stacked=True, ax=ax, color=LOC_COLORS, width=0.65)
ax.set_ylabel("Proportion of calls", fontsize=FS)
ax.set_xlabel("")
ax.set_xticklabels(CATEGORIES, rotation=0, fontsize=FS)
ax.tick_params(axis="y", labelsize=FS)
ax.set_ylim(0, 1)
ax.set_yticks([0,1])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
leg = ax.legend(title="Location", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=FS)
leg.get_title().set_fontsize(FS)
for i, n in enumerate(totals):
    ax.text(i, 1.02, f"n={int(n):,}", ha="center", fontsize=FS)
fig.suptitle("Location split by call context  (all dates pooled)", y=1.04, fontsize=FS)
fig.tight_layout()
save_fig(fig, "location_split_warble_bout_inword_alarm")
# Also save a copy into the repo so it's easy to download to a local machine.
import os
_repo_fig_dir = Path(os.getcwd()) / "figures"
_repo_fig_dir.mkdir(parents=True, exist_ok=True)
_pdf_path = _repo_fig_dir / "location_split_warble_bout_inword_alarm.pdf"
fig.savefig(_pdf_path, bbox_inches="tight")
print(f"Saved PDF to {_pdf_path}")

plt.show()
